# 4.22 Non-negative Matrix Factorization

Non-negative Matrix Factorization turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Part 4 moves from prediction with labels to discovery without labels. Matrix factorization becomes parts-based when every factor is constrained to be non-negative. NMF is not PCA: it cannot use negative cancellation, so preprocessing must preserve nonnegative inputs.

Save a copy to Drive to edit.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def sample_for_embedding(X, y, max_points=500, seed=SEED):
    rng = np.random.default_rng(seed)
    if X.shape[0] <= max_points:
        return X, y
    idx = rng.choice(X.shape[0], size=max_points, replace=False)
    order = np.sort(idx)
    return X[order], y[order]


def standardize_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def nonnegative_for_factorization(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)


def safe_trustworthiness(X, Z):
    neighbors = min(10, max(1, (X.shape[0] - 1) // 3))
    return float(trustworthiness(X, Z, n_neighbors=neighbors))


def plot_ladder_embeddings(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(17, 3.4))
    for ax, item in zip(axes, results):
        scatter = ax.scatter(item["Z"][:, 0], item["Z"][:, 1], c=item["y"], s=10, cmap="viridis")
        ax.set_title(item["name"].split("(")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("2D embeddings across the D1-D5 ladder")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"D{i}" for i in xs])
    ax.set_ylabel(metric_name)
    ax.set_xlabel("dataset rung")
    ax.set_title(f"{metric_name} vs. ladder complexity")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        unique = np.unique(y)
        label_info = len(unique) if unique.size < 30 else "continuous"
        print(f"D{i}: {name} | X={X.shape} | color/label info={label_info}")
        print(np.round(X[:3, :min(5, X.shape[1])], 3))

## The concept, built once: SVD baseline versus nonnegative factors

The lesson's generic low-rank audit is $$X_c=U\Sigma V^	op,\qquad Z=X_cV_r$$, with first-component energy $0.800$ on the toy matrix. NMF is different: it minimizes reconstruction error under nonnegativity constraints, so we keep inputs nonnegative instead of centering them.

In [ ]:
toy = np.array([[1.0, 2.0], [2.0, 1.0], [3.0, 4.0], [4.0, 3.0]])
means = toy.mean(axis=0)
centered = toy - means
_, singular_values, vt = np.linalg.svd(centered, full_matrices=False)
energies = singular_values ** 2
first_energy = energies[0] / energies.sum()
print("SVD baseline energy", first_energy)
assert np.allclose(means, [2.5, 2.5])
assert np.allclose(np.round(singular_values, 3), [2.828, 1.414])
assert np.isclose(first_energy, 0.8)
assert np.all(toy >= 0.0)

Now build the reusable method. The model learns $Xpprox WH$ with $W\ge 0$ and $H\ge 0$; the 2D code $W$ is used as the factor-space view and reconstruction error is the metric.

In [ ]:
def method(X_nonnegative, n_components=2, seed=SEED):
    X_nonnegative = np.asarray(X_nonnegative, dtype=float)
    if np.any(X_nonnegative < 0.0):
        raise ValueError("NMF requires nonnegative inputs")
    components = min(n_components, X_nonnegative.shape[1])
    model = NMF(n_components=components, init="nndsvda", random_state=seed, max_iter=600)
    W = model.fit_transform(X_nonnegative)
    H = model.components_
    reconstruction = W @ H
    error = float(np.linalg.norm(X_nonnegative - reconstruction) / np.linalg.norm(X_nonnegative))
    if W.shape[1] == 1:
        W = np.column_stack([W[:, 0], np.zeros(W.shape[0])])
    return W[:, :2], error, H

W_toy, toy_error, H_toy = method(toy, n_components=2, seed=0)
print("toy NMF code shape", W_toy.shape)
print("toy relative reconstruction error", round(toy_error, 6))
assert W_toy.shape == (4, 2)
assert toy_error >= 0.0

## The dataset ladder

We use the shared F3 dimensionality-reduction ladder: a near-1D toy line, a 3D swiss-roll-style surface, real handwritten digits, digits with added noise dimensions, and a real 30-dimensional breast-cancer feature table. The labels are only colors for visualization; the embedding method does not train on them.

In [ ]:
rungs = dimred_ladder()
preview_ladder(rungs)

## Run the same method across D1-D5

Each rung is standardized or scaled in the same way, embedded into 2D, and scored with relative reconstruction error. Subsampling is seeded and only bounds future notebook runtime; this build script does not execute the notebook.

In [ ]:
metric_name = "relative reconstruction error"
results = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    X_small, y_small = sample_for_embedding(X, y, max_points=600, seed=SEED + rung_index)
    X_nonnegative = nonnegative_for_factorization(X_small)
    Z, error, H = method(X_nonnegative, n_components=2, seed=SEED + rung_index)
    results.append({"name": name, "X": X_nonnegative, "y": y_small, "Z": Z, "metric": error})
    print(f"D{rung_index} | {name:32s} | rel_reconstruction_error={error:.3f}")

## Results visualization

The closing figure has two parts: small-multiple embedding panels for D1-D5, then a metric curve as the data become more realistic and higher dimensional.

In [ ]:
plot_ladder_embeddings(results, metric_name)

## Pitfall on the hardest rung: negative preprocessing breaks NMF

The lesson warns that scale and preprocessing control the computation. Centering creates negative values, which violates NMF's contract; the fix is nonnegative scaling rather than PCA-style centering.

In [ ]:
name, X, y = rungs[-1]
X_small, y_small = sample_for_embedding(X, y, max_points=600, seed=49)
centered = X_small - X_small.mean(axis=0)
try:
    method(centered, n_components=2, seed=0)
except ValueError as exc:
    print("wrong preprocessing failed:", exc)
X_fixed = nonnegative_for_factorization(X_small)
Z_fixed, fixed_error, H_fixed = method(X_fixed, n_components=2, seed=0)
print("fixed nonnegative min", round(float(X_fixed.min()), 3))
print("fixed reconstruction error", round(fixed_error, 3))

## Evaluate it + Practice

- Report relative reconstruction error next to a no-skill baseline such as plotting two raw standardized features or a random 2D map.
- Sanity check that nearby points in the original space still have nearby points in the embedding.
- Ablate the key idea: remove scaling, change the random seed, or use too few neighbors/components and watch the metric move.
- Watch for failure signals: unstable layouts, one feature dominating distances, disconnected neighbor graphs, or a better objective with worse held-out structure.
- Treat labels as a post-hoc audit only; unsupervised methods do not get to train on them.


### Practice

Compare NMF with PCA reconstruction error on D4 after scaling to [0, 1].

In [ ]:
# Your code here


### Practice

Increase NMF components from 2 to 5 on D5 and record train reconstruction error.

In [ ]:
# Your code here


### Practice

Inspect the largest entries of each learned component on D5 and explain why they are not ground truth factors.

In [ ]:
# Your code here
